In [ ]:
# Cloning the repository to get the model definitions
!git clone https://github.com/waterdisappear/SARATR-X.git
import sys
sys.path.append('/kaggle/working/SARATR-X/pre-training')

Cloning into 'SARATR-X'...
remote: Enumerating objects: 2304, done.
remote: Counting objects: 100% (2304/2304), done.
remote: Compressing objects: 100% (1627/1627), done.
remote: Total 2304 (delta 686), reused 2220 (delta 636), pack-reused 0 (from 0)
Receiving objects: 100% (2304/2304), 37.54 MiB | 24.91 MiB/s, done.
Resolving deltas: 100% (686/686), done.


In [2]:
!pip install timm==0.5.4 huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 11.9 MB/s eta 0:00:0000:01
  Attempting uninstall: timm
    Found existing installation: timm 1.0.25
    Uninstalling timm-1.0.25:
      Successfully uninstalled timm-1.0.25


In [4]:
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Correct repo and path based on your list
checkpoint_path = hf_hub_download(
    repo_id="waterdisappear/SARATR-X",
    filename="mae_hivit_base_1600ep.pth",
    subfolder="pre-training", 
    token=hf_token
)

print(f"Success! Weights are at: {checkpoint_path}")

pre-training/mae_hivit_base_1600ep.pth:   0%|          | 0.00/263M [00:00<?, ?B/s]

Success! Weights are at: /root/.cache/huggingface/hub/models--waterdisappear--SARATR-X/snapshots/32cc9ecca6c33832579b189221841cc52e4d45e3/pre-training/mae_hivit_base_1600ep.pth


In [7]:
sys.path.append('/kaggle/working/SARATR-X/pre-training/models')

SARATR_CKPT = "/root/.cache/huggingface/hub/models--waterdisappear--SARATR-X/snapshots/32cc9ecca6c33832579b189221841cc52e4d45e3/pre-training/mae_hivit_base_1600ep.pth"


In [ ]:
import os
import json
from pathlib import Path
from sklearn.model_selection import train_test_split

# --- Configuration ---
DATA_ROOT = Path("/kaggle/input/datasets/harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset/data")
CLASS_0_DIR = DATA_ROOT / "S1SAR_UnBalanced_400by400_Class_0" / "0"
CLASS_1_DIR = DATA_ROOT / "S1SAR_UnBalanced_400by400_Class_1" / "1"

def build_dataset_metadata():
    all_samples = []
    
    # Audits Class 0 (No Spill)
    class_0_files = list(CLASS_0_DIR.glob("*.jpg"))
    for f in class_0_files:
        all_samples.append({"path": str(f), "label": 0})
        
    # Audits Class 1 (Oil Spill)
    class_1_files = list(CLASS_1_DIR.glob("*.jpg"))
    for f in class_1_files:
        all_samples.append({"path": str(f), "label": 1})
    
    print(f"--- Dataset Audit ---")
    print(f"Class 0 (No Spill): {len(class_0_files)} images")
    print(f"Class 1 (Oil Spill): {len(class_1_files)} images")
    print(f"Total Images: {len(all_samples)}")
    
    return all_samples

# 1. Collects all paths and labels
samples = build_dataset_metadata()
paths = [s['path'] for s in samples]
labels = [s['label'] for s in samples]

# 2. First Split: 80% Train + Val, 20% Test (Stratified)
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    paths, labels, test_size=0.2, stratify=labels, random_state=42
)

# 3. Second Split: 15% of the remainder for Validation (~12% of total)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, test_size=0.15, stratify=train_val_labels, random_state=42
)

# 4. Saving splits to JSON 
splits = {
    "train": [{"path": p, "label": l} for p, l in zip(train_paths, train_labels)],
    "val": [{"path": p, "label": l} for p, l in zip(val_paths, val_labels)],
    "test": [{"path": p, "label": l} for p, l in zip(test_paths, test_labels)]
}

with open("/kaggle/working/dataset_splits.json", "w") as f:
    json.dump(splits, f)

print(f"\n--- Splitting Results ---")
print(f"Training:   {len(splits['train'])} samples")
print(f"Validation: {len(splits['val'])} samples")
print(f"Testing:    {len(splits['test'])} samples")

--- Dataset Audit ---
Class 0 (No Spill): 3725 images
Class 1 (Oil Spill): 1905 images
Total Images: 5630

--- Splitting Results ---
Training:   3828 samples
Validation: 676 samples
Testing:    1126 samples


In [9]:
!mkdir -p /kaggle/working/code
!mkdir -p /kaggle/working/CROMA

In [10]:
%%writefile /kaggle/working/code/dataset.py
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T
import numpy as np

class OilSpillSARDataset(Dataset):
    def __init__(self, samples, img_size=224, split='train'):
        """
        samples: list of dicts from dataset_splits.json
        """
        self.samples = samples
        self.img_size = img_size
        self.split = split
        
        # SARATR-X specific normalization and augmentation
        if split == 'train':
            self.transform = T.Compose([
                T.Resize((img_size, img_size)),
                T.RandomHorizontalFlip(),
                T.RandomVerticalFlip(),
                T.ToTensor(),
                T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])
        else:
            self.transform = T.Compose([
                T.Resize((img_size, img_size)),
                T.ToTensor(),
                T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        img_path = item['path']
        label = item['label']

        # Load SAR image (JPG format from dataset)
        # We convert to RGB to provide the 3 channels SARATR-X expects
        img = Image.open(img_path).convert('RGB')
        
        if self.transform:
            img = self.transform(img)
            
        return img, torch.tensor(label, dtype=torch.long)

Writing /kaggle/working/code/dataset.py


In [ ]:
%%writefile /kaggle/working/code/model_for_saratr.py
import torch
import torch.nn as nn
import sys
sys.path.append('/kaggle/working/SARATR-X/pre-training/models')
from models_hivit import hivit_base

class SARATROilSpillClassifier(nn.Module):
    def __init__(self, weight_path, num_classes=2):
        super().__init__()
        self.backbone = hivit_base(img_size=224)
        
        # Loading pre-trained weights
        checkpoint = torch.load(weight_path, map_location='cpu')
        state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint
        self.backbone.load_state_dict(state_dict, strict=False)

        # Freezing backbone for stable training
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        # Classification Head: Input is 512 (HiViT base dim)
        self.head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # features shape: [Batch, Tokens, 512]
        features = self.backbone.forward_features(x)
        
        # Global Average Pooling across tokens to get image representation
        cls_feature = features.mean(dim=1) 
        
        return self.head(cls_feature)

Writing /kaggle/working/code/model_for_saratr.py


In [ ]:
%%writefile /kaggle/working/code/tune_oil_spill.py
import os
import json
import torch
import torch.nn as nn
import optuna
from torch.utils.data import DataLoader, Subset
from dataset import OilSpillSARDataset
from model_for_saratr import SARATROilSpillClassifier

# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SARATR_CKPT = "/root/.cache/huggingface/hub/models--waterdisappear--SARATR-X/snapshots/32cc9ecca6c33832579b189221841cc52e4d45e3/pre-training/mae_hivit_base_1600ep.pth"
SPLITS_JSON = "/kaggle/working/dataset_splits.json"

def objective(trial):
    # 1. Suggests Hyperparameters
    lr = trial.suggest_float("lr", 1e-5, 5e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 0.1, log=True)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)

    # 2. Loads Data
    with open(SPLITS_JSON, "r") as f:
        splits = json.load(f)

    # Uses a subset for faster tuning trials if desired
    train_ds = OilSpillSARDataset(splits['train'], split='train')
    val_ds = OilSpillSARDataset(splits['val'], split='val')

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    # 3. Initializes Model with suggested dropout
    model = SARATROilSpillClassifier(SARATR_CKPT, num_classes=2).to(DEVICE)
    # Updates dropout dynamically in the head
    model.head[2] = nn.Dropout(dropout)

    weights = torch.tensor([1.0, 2.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=lr, weight_decay=weight_decay)

    # 4. Short Training Loop 
    for epoch in range(10):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        accuracy = val_correct / val_total
        
        # Reporting back to Optuna for pruning poor trials
        trial.report(accuracy, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return accuracy

if __name__ == "__main__":
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=20) # Total number of combinations to try

    print("\n--- Tuning Results ---")
    print(f"Best Accuracy: {study.best_value:.4f}")
    print(f"Best Params: {json.dumps(study.best_params, indent=4)}")

Writing /kaggle/working/code/tune_oil_spill.py


In [15]:
!python /kaggle/working/code/tune_oil_spill.py

[I 2026-04-04 13:33:13,730] A new study created in memory with name: no-name-92559af4-2674-4d09-b66a-141f49141b0a
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
[I 2026-04-04 13:43:48,858] Trial 0 finished with value: 0.8668639053254438 and parameters: {'lr': 3.1997698865169164e-05, 'batch_size': 16, 'weight_decay': 0.00816373968755196, 'dropout': 0.42451546088092174}. Best is trial 0 with value: 0.8668639053254438.
[I 2026-04-04 13:54:11,964] Trial 1 finished with value: 0.9127218934911243 and parameters: {'lr': 0.0002634990178597008, 'batch_size': 32, 'weight_decay': 0.0025271424860255826, 'dropout': 0.48019210839691306}. Best is trial 1 with value: 0.9127218934911243.
[I 2026-04-04 14:04:54,389] Trial 2 finished with

In [ ]:
%%writefile /kaggle/working/code/train_oil_spill.py
import os
import json
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import DataLoader
from dataset import OilSpillSARDataset
from model_for_saratr import SARATROilSpillClassifier

# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SARATR_CKPT = "/root/.cache/huggingface/hub/models--waterdisappear--SARATR-X/snapshots/32cc9ecca6c33832579b189221841cc52e4d45e3/pre-training/mae_hivit_base_1600ep.pth"
SPLITS_JSON = "/kaggle/working/dataset_splits.json"
SAVE_PATH = "/kaggle/working/oil_spill_classifier_best.pt"

# Optimal Params from Tuning
BATCH_SIZE = 32
LR = 0.00030358
WEIGHT_DECAY = 0.0008399
DROPOUT = 0.3778
EPOCHS = 50

def train():
    # 1. Loads Splits
    with open(SPLITS_JSON, "r") as f:
        splits = json.load(f)

    train_ds = OilSpillSARDataset(splits['train'], split='train')
    val_ds = OilSpillSARDataset(splits['val'], split='val')

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    # 2. Initializes Model
    model = SARATROilSpillClassifier(SARATR_CKPT, num_classes=2).to(DEVICE)
    
    # Applies optimal dropout to the head
    model.head[2] = nn.Dropout(DROPOUT)

    # 3. Handles Imbalance with Weighted Loss
    weights = torch.tensor([1.0, 2.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    # Optimizes only the head
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_acc = 0.0

    for epoch in range(EPOCHS):
        model.train()
        train_loss, correct, total = 0, 0, 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for imgs, labels in pbar:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix({"Loss": f"{loss.item():.4f}", "Acc": f"{100.*correct/total:.2f}%"})

        scheduler.step()

        # --- Validation ---
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_acc = 100. * val_correct / val_total
        print(f"Validation Accuracy: {val_acc:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), SAVE_PATH)
            print(f">>> Saved Best Model (Acc: {best_acc:.2f}%)")

if __name__ == "__main__":
    train()

Overwriting /kaggle/working/code/train_oil_spill.py


In [17]:
!python /kaggle/working/code/train_oil_spill.py

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Epoch 1/50: 100%|████| 120/120 [00:40<00:00,  2.94it/s, Loss=0.6248, Acc=68.47%]
Validation Accuracy: 83.28%
>>> Saved Best Model (Acc: 83.28%)
Epoch 2/50: 100%|████| 120/120 [00:44<00:00,  2.72it/s, Loss=0.3172, Acc=81.56%]
Validation Accuracy: 87.43%
>>> Saved Best Model (Acc: 87.43%)
Epoch 3/50: 100%|████| 120/120 [00:49<00:00,  2.41it/s, Loss=0.1362, Acc=85.37%]
Validation Accuracy: 83.88%
Epoch 4/50: 100%|████| 120/120 [00:50<00:00,  2.37it/s, Loss=0.4171, Acc=88.77%]
Validation Accuracy: 85.50%
Epoch 5/50: 100%|████| 120/120 [00:51<00:00,  2.34it/s, Loss=0.4441, Acc=89.42%]
Validation Accuracy: 90.38%
>>> Saved Best Model (Acc: 90.38%)
Epoch 6/50: 100%|████| 120/120 [

In [ ]:
%%writefile /kaggle/working/code/evaluate_oil_spill.py
import torch
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from torch.utils.data import DataLoader
from dataset import OilSpillSARDataset
from model_for_saratr import SARATROilSpillClassifier

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SARATR_CKPT = "/root/.cache/huggingface/hub/models--waterdisappear--SARATR-X/snapshots/32cc9ecca6c33832579b189221841cc52e4d45e3/pre-training/mae_hivit_base_1600ep.pth"
WEIGHTS_PATH = "/kaggle/working/oil_spill_classifier_best.pt"
SPLITS_JSON = "/kaggle/working/dataset_splits.json"

def evaluate():
    # 1. Loads Test Split
    with open(SPLITS_JSON, "r") as f:
        splits = json.load(f)
    test_ds = OilSpillSARDataset(splits['test'], split='test')
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

    # 2. Loads Model
    model = SARATROilSpillClassifier(SARATR_CKPT, num_classes=2).to(DEVICE)
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
    model.eval()

    all_preds = []
    all_labels = []

    # 3. Runs Inference
    with torch.no_grad():
        for imgs, labels in test_loader:
            outputs = model(imgs.to(DEVICE))
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    # 4. Generates Reports
    print("\n" + "="*30)
    print("FINAL TEST SET REPORT")
    print("="*30)
    print(classification_report(all_labels, all_preds, target_names=['No Spill', 'Oil Spill']))
    
    # 5. Plots Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Spill', 'Oil Spill'], yticklabels=['No Spill', 'Oil Spill'])
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.title('SARATR-X Oil Spill Confusion Matrix')
    plt.savefig('/kaggle/working/confusion_matrix.png')
    print("\n>>> Confusion Matrix saved to /kaggle/working/confusion_matrix.png")

if __name__ == "__main__":
    evaluate()

Writing /kaggle/working/code/evaluate_oil_spill.py


In [19]:
!python /kaggle/working/code/evaluate_oil_spill.py


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]

FINAL TEST SET REPORT
              precision    recall  f1-score   support

    No Spill       0.97      0.96      0.97       745
   Oil Spill       0.92      0.94      0.93       381

    accuracy                           0.95      1126
   macro avg       0.95      0.95      0.95      1126
weighted avg       0.96      0.95      0.95      1126


>>> Confusion Matrix saved to /kaggle/working/confusion_matrix.png


In [ ]:
%%writefile /kaggle/working/code/visualize_results.py
import torch
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T
from model_for_saratr import SARATROilSpillClassifier

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SARATR_CKPT = "/root/.cache/huggingface/hub/models--waterdisappear--SARATR-X/snapshots/32cc9ecca6c33832579b189221841cc52e4d45e3/pre-training/mae_hivit_base_1600ep.pth"
WEIGHTS_PATH = "/kaggle/working/oil_spill_classifier_best.pt"
SPLITS_JSON = "/kaggle/working/dataset_splits.json"

def visualize():
    # 1. Loads Model
    model = SARATROilSpillClassifier(SARATR_CKPT, num_classes=2).to(DEVICE)
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
    model.eval()

    # 2. Loads Test Samples
    with open(SPLITS_JSON, "r") as f:
        test_samples = json.load(f)['test']
    
    # Picks 16 random samples
    selected = random.sample(test_samples, 16)
    
    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    fig, axes = plt.subplots(4, 4, figsize=(20, 20))
    axes = axes.flatten()

    for i, item in enumerate(selected):
        img_path = item['path']
        actual_label = "Oil Spill" if item['label'] == 1 else "No Spill"
        
        # Inference
        raw_img = Image.open(img_path).convert('RGB')
        img_tensor = transform(raw_img).unsqueeze(0).to(DEVICE)
        
        with torch.no_grad():
            output = model(img_tensor)
            prob = torch.softmax(output, dim=1)
            pred_idx = torch.argmax(prob, dim=1).item()
            confidence = prob[0][pred_idx].item()
            pred_label = "Oil Spill" if pred_idx == 1 else "No Spill"

        # Plotting the three-panel visualisation
        axes[i].imshow(raw_img, cmap='gray')
        color = 'green' if pred_label == actual_label else 'red'
        axes[i].set_title(f"Act: {actual_label}\nPred: {pred_label} ({confidence:.2%})", color=color)
        axes[i].axis('off')

    plt.tight_layout()
    plt.savefig('/kaggle/working/prediction_gallery.png')
    plt.show()

if __name__ == "__main__":
    visualize()

Writing /kaggle/working/code/visualize_results.py


In [21]:
!python /kaggle/working/code/visualize_results.py


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Figure(2000x2000)
